# Kankor Retrieval Benchmark: `BAAI/bge-m3`

This notebook benchmarks the canonical page corpus with the new `bge_m3` dense backend.

It supports:
- BGE-M3 dense retrieval
- BGE-M3 dense + lexical + RRF
- optional reranker evaluation
- reranker candidate-pool sweeps for overnight runs
- promotion analysis for whether a relevant answer found in the candidate pool gets moved into top 3
- optional direct comparison against the `text-embedding-3-large` baseline


In [1]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'packages' / 'rag_core').exists() and (candidate / 'apps' / 'api').exists():
            return candidate
    raise RuntimeError('Could not resolve repo root from notebook location.')

REPO_ROOT = resolve_repo_root(Path.cwd())
DOTENV_PATH = REPO_ROOT / 'docker' / '.env'
if DOTENV_PATH.exists():
    for raw_line in DOTENV_PATH.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

sys.path.insert(0, str(REPO_ROOT / 'packages' / 'rag_core' / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'apps' / 'api' / 'src'))

from rag_core.eval.retrieval_benchmark import (
    benchmark_dense_retrieval,
    benchmark_hybrid_retrieval,
    benchmark_hybrid_reranked_retrieval,
    load_qrels,
    load_query_suite,
)
from rag_core.impl.embeddings_bge_m3 import BGEM3Embedder
from rag_core.impl.embeddings_openai import OpenAIEmbedder
from rag_core.impl.embeddings_e5 import MultilingualE5Embedder

from rag_core.impl.reranker_onnx import ONNXSequenceClassificationReranker
from rag_core.impl.vector_faiss import FaissVectorStore
from rag_core.rag.retrieval import DenseRetriever, LexicalRetriever, PageRetrievalEngine, RRFFusionPolicy


In [2]:
QUERY_SUITE_NAME = 'chunking_pages_grade10_new_questions_suite_v1'
QRELS_NAME = 'chunking_pages_grade10_new_questions_qrels_v1'

#QUERY_SUITE_NAME = 'chunking_pages_suite_v2'
#QRELS_NAME = 'chunking_pages_qrels_v2'
RUN_OPENAI_BASELINE = False
RUN_RERANKER = True
QUERY_LIMIT = 200


K_VALUES = (1, 3, 5, 10)
QUERY_BATCH_SIZE = 64
RRF_K = 20
DENSE_MIN_SCORE = 0.15
LEXICAL_MIN_SCORE = 0.01
RERANKER_TARGET_TOP_K = 3
RERANKER_CANDIDATE_POOL_SIZES = (10,)
RERANKER_CANDIDATE_POOL_SIZE = 10
SAVE_RESULTS = True
RESULTS_ROOT = REPO_ROOT / 'data' / 'experiments' / 'benchmark_kankor_bge_m3'
PRIMARY_INDEX_DIR = REPO_ROOT / 'data' / 'index' / 'kankor_bge_m3_full'
PRIMARY_SYSTEM_NAME = 'bge_m3'

RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

OPENAI_INDEX_DIR = REPO_ROOT / 'data' / 'index' / 'kankor_openai_full'
QUERY_SUITE_PATH = REPO_ROOT / 'data' / 'query_suites' / f'{QUERY_SUITE_NAME}.jsonl'
QRELS_PATH = REPO_ROOT / 'data' / 'query_suites' / f'{QRELS_NAME}.jsonl'

RERANKER_MODEL_ID = os.getenv('RAG_RERANKER_MODEL_ID', 'onnx-community/gte-multilingual-reranker-base')
RERANKER_MODEL_REVISION = os.getenv('RAG_RERANKER_MODEL_REVISION') or None
RERANKER_MAX_LENGTH = int(os.getenv('RAG_RERANKER_MAX_LENGTH', '256'))
RERANKER_BATCH_SIZE = int(os.getenv('RAG_RERANKER_BATCH_SIZE', '16'))

assert (PRIMARY_INDEX_DIR / 'index.faiss').exists(), f'Missing primary index: {PRIMARY_INDEX_DIR / "index.faiss"}'
assert (PRIMARY_INDEX_DIR / 'metadata.jsonl').exists(), f'Missing primary metadata: {PRIMARY_INDEX_DIR / "metadata.jsonl"}'
assert QUERY_SUITE_PATH.exists(), f'Missing query suite: {QUERY_SUITE_PATH}'
assert QRELS_PATH.exists(), f'Missing qrels: {QRELS_PATH}'
if RUN_OPENAI_BASELINE:
    assert (OPENAI_INDEX_DIR / 'index.faiss').exists(), f'Missing OpenAI index: {OPENAI_INDEX_DIR / "index.faiss"}'
    assert (OPENAI_INDEX_DIR / 'metadata.jsonl').exists(), f'Missing OpenAI metadata: {OPENAI_INDEX_DIR / "metadata.jsonl"}'

display({
    'repo_root': str(REPO_ROOT),
    'primary_system_name': PRIMARY_SYSTEM_NAME,
    'primary_index_dir': str(PRIMARY_INDEX_DIR),
    'openai_index_dir': str(OPENAI_INDEX_DIR),
    'query_suite': str(QUERY_SUITE_PATH),
    'qrels': str(QRELS_PATH),
    'run_openai_baseline': RUN_OPENAI_BASELINE,
    'run_reranker': RUN_RERANKER,
    'reranker_target_top_k': RERANKER_TARGET_TOP_K,
    'reranker_candidate_pool_sizes': RERANKER_CANDIDATE_POOL_SIZES,
    'default_reranker_candidate_pool_size': RERANKER_CANDIDATE_POOL_SIZE,
    'save_results': SAVE_RESULTS,
    'results_root': str(RESULTS_ROOT),
    'run_label': RUN_LABEL,
})


{'repo_root': '/home/nasher/Documents/projects/kankor-rag-space',
 'primary_system_name': 'bge_m3',
 'primary_index_dir': '/home/nasher/Documents/projects/kankor-rag-space/data/index/kankor_bge_m3_full',
 'openai_index_dir': '/home/nasher/Documents/projects/kankor-rag-space/data/index/kankor_openai_full',
 'query_suite': '/home/nasher/Documents/projects/kankor-rag-space/data/query_suites/chunking_pages_grade10_new_questions_suite_v1.jsonl',
 'qrels': '/home/nasher/Documents/projects/kankor-rag-space/data/query_suites/chunking_pages_grade10_new_questions_qrels_v1.jsonl',
 'run_openai_baseline': False,
 'run_reranker': True,
 'reranker_target_top_k': 3,
 'reranker_candidate_pool_sizes': (10,),
 'default_reranker_candidate_pool_size': 10,
 'save_results': True,
 'results_root': '/home/nasher/Documents/projects/kankor-rag-space/data/experiments/benchmark_kankor_bge_m3',
 'run_label': '20260409T164725Z'}

In [3]:
queries = load_query_suite(QUERY_SUITE_PATH, allowed_intents={'grounded_textbook'})
qrels_by_query = load_qrels(QRELS_PATH)
if QUERY_LIMIT is not None:
    queries = queries[: int(QUERY_LIMIT)]

def build_stack(index_dir: Path, *, embedder):
    vector_store = FaissVectorStore.load(
        index_path=index_dir / 'index.faiss',
        metadata_path=index_dir / 'metadata.jsonl',
    )
    dense = DenseRetriever(embedder=embedder, vector_store=vector_store, min_score=DENSE_MIN_SCORE)
    lexical = LexicalRetriever(vector_store=vector_store, min_score=LEXICAL_MIN_SCORE)
    engine = PageRetrievalEngine(
        dense_retriever=dense,
        lexical_retriever=lexical,
        fusion_policy=RRFFusionPolicy(rrf_k=RRF_K),
    )
    return vector_store, dense, engine

primary_embedder = BGEM3Embedder(
    model_name=os.getenv('RAG_BGE_M3_MODEL_ID', 'BAAI/bge-m3'),
    batch_size=int(os.getenv('RAG_BGE_M3_BATCH_SIZE', '32')),
    use_fp16=(os.getenv('RAG_BGE_M3_USE_FP16') or '').strip().lower() == 'true' if os.getenv('RAG_BGE_M3_USE_FP16') is not None else None,
    device=os.getenv('RAG_BGE_M3_DEVICE') or None,
    max_length=int(os.getenv('RAG_BGE_M3_MAX_LENGTH', '8192')),
)
_, primary_dense_retriever, primary_hybrid_engine = build_stack(PRIMARY_INDEX_DIR, embedder=primary_embedder)

openai_dense_retriever = None
openai_hybrid_engine = None
if RUN_OPENAI_BASELINE:
    openai_embedder = OpenAIEmbedder(
        model_name=os.getenv('RAG_OPENAI_EMBEDDING_MODEL_ID', 'text-embedding-3-large'),
        api_key=os.getenv('RAG_OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY'),
        base_url=os.getenv('RAG_OPENAI_BASE_URL') or None,
        dimensions=int(os.getenv('RAG_OPENAI_EMBEDDING_DIMENSIONS')) if os.getenv('RAG_OPENAI_EMBEDDING_DIMENSIONS') else None,
        timeout_seconds=float(os.getenv('RAG_OPENAI_TIMEOUT_SECONDS', '120.0')),
    )
    _, openai_dense_retriever, openai_hybrid_engine = build_stack(OPENAI_INDEX_DIR, embedder=openai_embedder)

reranker = None
if RUN_RERANKER:
    reranker = ONNXSequenceClassificationReranker(
        model_name=RERANKER_MODEL_ID,
        model_revision=RERANKER_MODEL_REVISION,
        max_length=RERANKER_MAX_LENGTH,
        batch_size=RERANKER_BATCH_SIZE,
    )
    reranker.warmup()

display({'queries': len(queries), 'qrels': len(qrels_by_query)})


/home/nasher/miniconda3/envs/kankor-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'queries': 200, 'qrels': 4410}

In [4]:
RERANKER_CANDIDATE_POOL_SIZES = tuple(
    sorted(
        {
            max(RERANKER_TARGET_TOP_K, int(value))
            for value in RERANKER_CANDIDATE_POOL_SIZES
        }
    )
)

DEFAULT_RERANKER_CANDIDATE_POOL_SIZE = max(RERANKER_TARGET_TOP_K, int(RERANKER_CANDIDATE_POOL_SIZE))
BASE_K_VALUES = tuple(sorted({*K_VALUES, RERANKER_TARGET_TOP_K}))
DEFAULT_RERANK_K_VALUES = tuple(sorted({*BASE_K_VALUES, DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}))

primary_dense_results = benchmark_dense_retrieval(
    queries=queries,
    qrels_by_query=qrels_by_query,
    dense_retriever=primary_dense_retriever,
    k_values=BASE_K_VALUES,
    query_batch_size=QUERY_BATCH_SIZE,
)
primary_hybrid_results = benchmark_hybrid_retrieval(
    queries=queries,
    qrels_by_query=qrels_by_query,
    retrieval_engine=primary_hybrid_engine,
    k_values=BASE_K_VALUES,
    query_batch_size=QUERY_BATCH_SIZE,
)
primary_reranked_results = None
primary_reranked_results_by_pool = {}
if RUN_RERANKER:
    print(
        f'Running default reranked benchmark for candidate_pool_size={DEFAULT_RERANKER_CANDIDATE_POOL_SIZE} '
        f'with k_values={DEFAULT_RERANK_K_VALUES} ...'
    )
    primary_reranked_results = benchmark_hybrid_reranked_retrieval(
        queries=queries,
        qrels_by_query=qrels_by_query,
        retrieval_engine=primary_hybrid_engine,
        reranker=reranker,
        k_values=DEFAULT_RERANK_K_VALUES,
        reranker_candidate_pool_size=DEFAULT_RERANKER_CANDIDATE_POOL_SIZE,
        query_batch_size=QUERY_BATCH_SIZE,
    )
    primary_reranked_results_by_pool[DEFAULT_RERANKER_CANDIDATE_POOL_SIZE] = primary_reranked_results
    for candidate_pool_size in RERANKER_CANDIDATE_POOL_SIZES:
        if candidate_pool_size == DEFAULT_RERANKER_CANDIDATE_POOL_SIZE:
            continue
        sweep_k_values = tuple(sorted({RERANKER_TARGET_TOP_K, 5, candidate_pool_size}))
        print(
            f'Running reranker sweep for candidate_pool_size={candidate_pool_size} '
            f'with k_values={sweep_k_values} ...'
        )
        primary_reranked_results_by_pool[candidate_pool_size] = benchmark_hybrid_reranked_retrieval(
            queries=queries,
            qrels_by_query=qrels_by_query,
            retrieval_engine=primary_hybrid_engine,
            reranker=reranker,
            k_values=sweep_k_values,
            reranker_candidate_pool_size=candidate_pool_size,
            query_batch_size=QUERY_BATCH_SIZE,
        )

openai_dense_results = None
openai_hybrid_results = None
if RUN_OPENAI_BASELINE:
    openai_dense_results = benchmark_dense_retrieval(
        queries=queries,
        qrels_by_query=qrels_by_query,
        dense_retriever=openai_dense_retriever,
        k_values=BASE_K_VALUES,
        query_batch_size=QUERY_BATCH_SIZE,
    )
    openai_hybrid_results = benchmark_hybrid_retrieval(
        queries=queries,
        qrels_by_query=qrels_by_query,
        retrieval_engine=openai_hybrid_engine,
        k_values=BASE_K_VALUES,
        query_batch_size=QUERY_BATCH_SIZE,
    )

def aggregate_frame(result, prefix: str):
    frame = pd.DataFrame.from_dict(result['aggregate_by_k'], orient='index').set_index('k')
    return frame.add_prefix(prefix)

comparison = aggregate_frame(primary_dense_results, f'{PRIMARY_SYSTEM_NAME}_dense_').join(aggregate_frame(primary_hybrid_results, f'{PRIMARY_SYSTEM_NAME}_hybrid_'))
if primary_reranked_results is not None:
    comparison = comparison.join(aggregate_frame(primary_reranked_results, f'{PRIMARY_SYSTEM_NAME}_reranked_'))
if openai_dense_results is not None:
    comparison = comparison.join(aggregate_frame(openai_dense_results, 'openai_dense_'))
if openai_hybrid_results is not None:
    comparison = comparison.join(aggregate_frame(openai_hybrid_results, 'openai_hybrid_'))
comparison.loc[list(K_VALUES)]


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 389.43it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00, 15.87it/s]


Running default reranked benchmark for candidate_pool_size=10 with k_values=(1, 3, 5, 10) ...


Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00, 15.96it/s]


,bge_m3_dense_query_count_evaluated,bge_m3_dense_query_count_skipped_no_hits,bge_m3_dense_mean_hit_rate_at_k,bge_m3_dense_mean_recall_at_k,bge_m3_dense_mean_mrr_at_k,bge_m3_dense_mean_ndcg_at_k,bge_m3_hybrid_query_count_evaluated,bge_m3_hybrid_query_count_skipped_no_hits,bge_m3_hybrid_mean_hit_rate_at_k,bge_m3_hybrid_mean_recall_at_k,bge_m3_hybrid_mean_mrr_at_k,bge_m3_hybrid_mean_ndcg_at_k,bge_m3_reranked_query_count_evaluated,bge_m3_reranked_query_count_skipped_no_hits,bge_m3_reranked_mean_hit_rate_at_k,bge_m3_reranked_mean_recall_at_k,bge_m3_reranked_mean_mrr_at_k,bge_m3_reranked_mean_ndcg_at_k
k,,,,,,,,,,,,,,,,,,
1,200,0,0.800,0.7450,0.800000,0.800000,200,0,0.795,0.7350,0.79500,0.795000,200,0,0.780,0.6925,0.780000,0.780000
3,200,0,0.935,0.9075,0.860000,0.576065,200,0,0.885,0.8300,0.83000,0.535919,200,0,0.915,0.8800,0.844167,0.580553
5,200,0,0.950,0.9250,0.863250,0.582263,200,0,0.930,0.9025,0.83975,0.562548,200,0,0.935,0.9150,0.848917,0.593487
10,200,0,0.985,0.9600,0.867722,0.593000,200,0,0.950,0.9300,0.84229,0.570530,200,0,0.950,0.9300,0.850756,0.598341


In [5]:
def per_query_frame(result, system_name: str, *, candidate_pool_size: int | None = None):
    frame = pd.DataFrame(result['per_query']).copy()
    frame['system'] = system_name
    frame['expected_subject'] = frame['expected_subjects'].apply(lambda values: values[0] if values else '')
    frame['candidate_pool_size'] = candidate_pool_size
    return frame

frames = [
    per_query_frame(primary_dense_results, f'{PRIMARY_SYSTEM_NAME}_dense'),
    per_query_frame(primary_hybrid_results, f'{PRIMARY_SYSTEM_NAME}_hybrid'),
]
if primary_reranked_results is not None:
    frames.append(per_query_frame(primary_reranked_results, f'{PRIMARY_SYSTEM_NAME}_reranked_pool_{DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}', candidate_pool_size=DEFAULT_RERANKER_CANDIDATE_POOL_SIZE))
for candidate_pool_size, result in primary_reranked_results_by_pool.items():
    system_name = f'{PRIMARY_SYSTEM_NAME}_reranked_pool_{candidate_pool_size}'
    if system_name == f'{PRIMARY_SYSTEM_NAME}_reranked_pool_{DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}':
        continue
    frames.append(per_query_frame(result, system_name, candidate_pool_size=candidate_pool_size))
if openai_dense_results is not None:
    frames.append(per_query_frame(openai_dense_results, 'openai_dense'))
if openai_hybrid_results is not None:
    frames.append(per_query_frame(openai_hybrid_results, 'openai_hybrid'))
all_per_query = pd.concat(frames, ignore_index=True)

def aggregate_for_display(result, *, label: str, k: int) -> dict:
    aggregate = result['aggregate_by_k'][str(k)]
    return {
        'system': label,
        'k': k,
        'mean_hit_rate_at_k': aggregate['mean_hit_rate_at_k'],
        'mean_mrr_at_k': aggregate['mean_mrr_at_k'],
        'mean_recall_at_k': aggregate['mean_recall_at_k'],
        'mean_ndcg_at_k': aggregate['mean_ndcg_at_k'],
        'query_count_evaluated': aggregate['query_count_evaluated'],
    }

reranker_sweep_rows = []
promotion_detail_frames = []
if RUN_RERANKER and primary_reranked_results_by_pool:
    hybrid_per_query = per_query_frame(primary_hybrid_results, f'{PRIMARY_SYSTEM_NAME}_hybrid')
    available_rank_columns = set(hybrid_per_query.columns)
    hybrid_pool_rank_columns = [
        f'first_relevant_rank_at_{pool_size}'
        for pool_size in RERANKER_CANDIDATE_POOL_SIZES
        if pool_size != RERANKER_TARGET_TOP_K and f'first_relevant_rank_at_{pool_size}' in available_rank_columns
    ]
    hybrid_rank_columns = ['query_id', 'query', 'language', 'expected_subject', f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}', *hybrid_pool_rank_columns]
    hybrid_rank_view = hybrid_per_query[hybrid_rank_columns].copy()

    for candidate_pool_size, result in primary_reranked_results_by_pool.items():
        label = f'{PRIMARY_SYSTEM_NAME}_reranked_pool_{candidate_pool_size}'
        reranked_per_query = per_query_frame(result, label, candidate_pool_size=candidate_pool_size)
        top3_rank_column = f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}'
        pool_rank_column = f'first_relevant_rank_at_{candidate_pool_size}'
        if pool_rank_column not in hybrid_rank_view.columns or pool_rank_column not in reranked_per_query.columns:
            continue
        reranked_rank_view = reranked_per_query[['query_id', top3_rank_column, pool_rank_column]].rename(
            columns={
                top3_rank_column: 'reranked_first_relevant_rank_at_top3',
                pool_rank_column: 'reranked_first_relevant_rank_at_pool',
            }
        )
        promotion_frame = hybrid_rank_view.merge(reranked_rank_view, on='query_id', how='left')
        promotion_frame['candidate_pool_size'] = candidate_pool_size
        promotion_frame['relevant_in_hybrid_pool'] = promotion_frame[f'first_relevant_rank_at_{candidate_pool_size}'].notna()
        promotion_frame['relevant_in_hybrid_top3'] = promotion_frame[f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}'].notna()
        promotion_frame['relevant_in_reranked_top3'] = promotion_frame['reranked_first_relevant_rank_at_top3'].notna()
        promotion_frame['promoted_into_top3'] = (
            promotion_frame['relevant_in_hybrid_pool']
            & ~promotion_frame['relevant_in_hybrid_top3']
            & promotion_frame['relevant_in_reranked_top3']
        )
        promotion_frame['lost_from_top3'] = (
            promotion_frame['relevant_in_hybrid_top3']
            & ~promotion_frame['relevant_in_reranked_top3']
        )
        promotion_frame['net_top3_gain'] = promotion_frame['relevant_in_reranked_top3'].astype(int) - promotion_frame['relevant_in_hybrid_top3'].astype(int)
        promotion_detail_frames.append(promotion_frame)

        candidate_mask = promotion_frame['relevant_in_hybrid_pool']
        promotion_candidate_mask = promotion_frame['relevant_in_hybrid_pool'] & ~promotion_frame['relevant_in_hybrid_top3']
        rerank_ms = pd.DataFrame(result['retrieval_rows'])['rerank_ms'] if result['retrieval_rows'] else pd.Series(dtype='float64')
        aggregate_top3 = result['aggregate_by_k'][str(RERANKER_TARGET_TOP_K)]
        aggregate_top5 = result['aggregate_by_k'][str(5)]
        reranker_sweep_rows.append({
            'candidate_pool_size': candidate_pool_size,
            'mean_hit_rate_at_3': aggregate_top3['mean_hit_rate_at_k'],
            'mean_mrr_at_3': aggregate_top3['mean_mrr_at_k'],
            'mean_recall_at_3': aggregate_top3['mean_recall_at_k'],
            'mean_hit_rate_at_5': aggregate_top5['mean_hit_rate_at_k'],
            'mean_mrr_at_5': aggregate_top5['mean_mrr_at_k'],
            'mean_ndcg_at_5': aggregate_top5['mean_ndcg_at_k'],
            'mean_rerank_ms': float(rerank_ms.mean()) if not rerank_ms.empty else None,
            'median_rerank_ms': float(rerank_ms.median()) if not rerank_ms.empty else None,
            'queries_with_relevant_in_pool': int(candidate_mask.sum()),
            'queries_with_relevant_in_pool_and_not_top3': int(promotion_candidate_mask.sum()),
            'pool_to_top3_conversion_rate': float(promotion_frame.loc[candidate_mask, 'relevant_in_reranked_top3'].mean()) if candidate_mask.any() else None,
            'promotion_rate_given_not_hybrid_top3': float(promotion_frame.loc[promotion_candidate_mask, 'promoted_into_top3'].mean()) if promotion_candidate_mask.any() else None,
            'promoted_query_count': int(promotion_frame['promoted_into_top3'].sum()),
            'lost_top3_query_count': int(promotion_frame['lost_from_top3'].sum()),
            'net_top3_gain': int(promotion_frame['net_top3_gain'].sum()),
        })

reranker_sweep_summary = pd.DataFrame(reranker_sweep_rows)
if not reranker_sweep_summary.empty:
    reranker_sweep_summary = reranker_sweep_summary.sort_values('candidate_pool_size')
promotion_details = pd.concat(promotion_detail_frames, ignore_index=True) if promotion_detail_frames else pd.DataFrame()

language_slice = (
    all_per_query.groupby(['system', 'language'], dropna=False)
    .agg(
        query_count=('query_id', 'count'),
        mean_hit_rate_at_5=('hit_rate_at_5', 'mean'),
        mean_mrr_at_5=('mrr_at_5', 'mean'),
        mean_recall_at_5=('recall_at_5', 'mean'),
    )
    .reset_index()
    .sort_values(['system', 'language'])
)

subject_failure_slice = (
    all_per_query.groupby(['system', 'expected_subject'], dropna=False)
    .agg(
        query_count=('query_id', 'count'),
        fail_count_at_5=('hit_rate_at_5', lambda values: int((1.0 - values).sum())),
        fail_rate_at_5=('hit_rate_at_5', lambda values: float(1.0 - values.mean())),
        miss_top10_count=('hit_rate_at_10', lambda values: int((1.0 - values).sum())),
        miss_top10_rate=('hit_rate_at_10', lambda values: float(1.0 - values.mean())),
        mean_hit_rate_at_5=('hit_rate_at_5', 'mean'),
        mean_mrr_at_5=('mrr_at_5', 'mean'),
        mean_recall_at_5=('recall_at_5', 'mean'),
    )
    .reset_index()
    .sort_values(['system', 'fail_rate_at_5'], ascending=[True, False])
)

failure_rows = all_per_query[all_per_query['hit_rate_at_5'] < 1.0].sort_values(['system', 'language', 'expected_subject', 'query_id'])
aggregate_snapshot = pd.DataFrame([
    aggregate_for_display(primary_dense_results, label=f'{PRIMARY_SYSTEM_NAME}_dense', k=5),
    aggregate_for_display(primary_hybrid_results, label=f'{PRIMARY_SYSTEM_NAME}_hybrid', k=5),
    *[
        aggregate_for_display(result, label=f'{PRIMARY_SYSTEM_NAME}_reranked_pool_{candidate_pool_size}', k=5)
        for candidate_pool_size, result in sorted(primary_reranked_results_by_pool.items())
    ],
])

display(aggregate_snapshot)
display(reranker_sweep_summary)
display(language_slice)
display(subject_failure_slice.head(30))
display(failure_rows[['system', 'query_id', 'language', 'expected_subject', 'query', 'first_relevant_rank_at_10']].head(100))
if not promotion_details.empty:
    promoted_examples = promotion_details[promotion_details['promoted_into_top3']].sort_values(['candidate_pool_size', 'language', 'expected_subject', 'query_id'])
    display(promoted_examples[['candidate_pool_size', 'query_id', 'language', 'expected_subject', 'query', f'first_relevant_rank_at_{RERANKER_TARGET_TOP_K}', 'reranked_first_relevant_rank_at_top3']].head(100))

summary_rows = []
if openai_dense_results is not None:
    primary_hit5 = comparison.loc[5, f'{PRIMARY_SYSTEM_NAME}_dense_mean_hit_rate_at_k']
    openai_hit5 = comparison.loc[5, 'openai_dense_mean_hit_rate_at_k']
    primary_mrr5 = comparison.loc[5, f'{PRIMARY_SYSTEM_NAME}_dense_mean_mrr_at_k']
    openai_mrr5 = comparison.loc[5, 'openai_dense_mean_mrr_at_k']
    primary_ps_hit5 = language_slice[(language_slice['system'] == f'{PRIMARY_SYSTEM_NAME}_dense') & (language_slice['language'] == 'ps')]['mean_hit_rate_at_5'].iloc[0]
    openai_ps_hit5 = language_slice[(language_slice['system'] == 'openai_dense') & (language_slice['language'] == 'ps')]['mean_hit_rate_at_5'].iloc[0]
    summary_rows.append({
        'comparison': f'{PRIMARY_SYSTEM_NAME}_dense_vs_openai_dense',
        'delta_hit_rate_at_5': primary_hit5 - openai_hit5,
        'delta_mrr_at_5': primary_mrr5 - openai_mrr5,
        'delta_pashto_hit_rate_at_5': primary_ps_hit5 - openai_ps_hit5,
        'promotion_candidate': bool((primary_hit5 > openai_hit5) and (primary_ps_hit5 > openai_ps_hit5)),
    })
baseline_summary = pd.DataFrame(summary_rows) if summary_rows else pd.DataFrame()
display(baseline_summary)

saved_paths = {}
if SAVE_RESULTS:
    run_output_dir = RESULTS_ROOT / RUN_LABEL
    run_output_dir.mkdir(parents=True, exist_ok=True)
    comparison.loc[list(K_VALUES)].to_csv(run_output_dir / 'aggregate_comparison.csv')
    aggregate_snapshot.to_csv(run_output_dir / 'aggregate_snapshot.csv', index=False)
    all_per_query.to_csv(run_output_dir / 'all_per_query.csv', index=False)
    language_slice.to_csv(run_output_dir / 'language_slice.csv', index=False)
    subject_failure_slice.to_csv(run_output_dir / 'subject_failure_slice.csv', index=False)
    failure_rows.to_csv(run_output_dir / 'failure_rows.csv', index=False)
    baseline_summary.to_csv(run_output_dir / 'baseline_summary.csv', index=False)
    if not reranker_sweep_summary.empty:
        reranker_sweep_summary.to_csv(run_output_dir / 'reranker_sweep_summary.csv', index=False)
    if not promotion_details.empty:
        promotion_details.to_csv(run_output_dir / 'reranker_promotion_details.csv', index=False)
    saved_paths = {
        'run_output_dir': str(run_output_dir),
        'aggregate_comparison': str(run_output_dir / 'aggregate_comparison.csv'),
        'reranker_sweep_summary': str(run_output_dir / 'reranker_sweep_summary.csv'),
        'reranker_promotion_details': str(run_output_dir / 'reranker_promotion_details.csv'),
    }
display(saved_paths)


,system,k,mean_hit_rate_at_k,mean_mrr_at_k,mean_recall_at_k,mean_ndcg_at_k,query_count_evaluated
0,bge_m3_dense,5,0.950,0.863250,0.9250,0.582263,200
1,bge_m3_hybrid,5,0.930,0.839750,0.9025,0.562548,200
2,bge_m3_reranked_pool_10,5,0.935,0.848917,0.9150,0.593487,200


,candidate_pool_size,mean_hit_rate_at_3,mean_mrr_at_3,mean_recall_at_3,mean_hit_rate_at_5,mean_mrr_at_5,mean_ndcg_at_5,mean_rerank_ms,median_rerank_ms,queries_with_relevant_in_pool,queries_with_relevant_in_pool_and_not_top3,pool_to_top3_conversion_rate,promotion_rate_given_not_hybrid_top3,promoted_query_count,lost_top3_query_count,net_top3_gain
0,10,0.915,0.844167,0.88,0.935,0.848917,0.593487,2851.08,2778.0,190,13,0.963158,0.692308,9,3,6


,system,language,query_count,mean_hit_rate_at_5,mean_mrr_at_5,mean_recall_at_5
0,bge_m3_dense,en,66,0.954545,0.878788,0.931818
1,bge_m3_dense,fa,67,0.970149,0.878109,0.947761
2,bge_m3_dense,ps,67,0.925373,0.833085,0.895522
3,bge_m3_hybrid,en,66,0.954545,0.850000,0.924242
4,bge_m3_hybrid,fa,67,0.970149,0.866667,0.947761
5,bge_m3_hybrid,ps,67,0.865672,0.802736,0.835821
6,bge_m3_reranked_pool_10,en,66,0.954545,0.857323,0.939394
7,bge_m3_reranked_pool_10,fa,67,0.970149,0.891045,0.947761
8,bge_m3_reranked_pool_10,ps,67,0.880597,0.798507,0.858209


,system,expected_subject,query_count,fail_count_at_5,fail_rate_at_5,miss_top10_count,miss_top10_rate,mean_hit_rate_at_5,mean_mrr_at_5,mean_recall_at_5
0,bge_m3_dense,computer_science,200,10,0.050,3,0.015,0.950,0.863250,0.9250
1,bge_m3_hybrid,computer_science,200,14,0.070,10,0.050,0.930,0.839750,0.9025
2,bge_m3_reranked_pool_10,computer_science,200,13,0.065,10,0.050,0.935,0.848917,0.9150


,system,query_id,language,expected_subject,query,first_relevant_rank_at_10
74,bge_m3_dense,chunk_v2_g10_dr_computer_q025_en,en,computer_science,How does distance learning change the educatio...,10.0
98,bge_m3_dense,chunk_v2_g10_dr_computer_q033_en,en,computer_science,Why does the text describe the internet as mak...,NaN
158,bge_m3_dense,chunk_v2_g10_dr_computer_q053_en,en,computer_science,"In the logic of this lesson, what role does a ...",8.0
96,bge_m3_dense,chunk_v2_g10_dr_computer_q033_fa,fa,computer_science,چرا متن، انترنت را عاملی برای شبیه شدن جهان به...,8.0
144,bge_m3_dense,chunk_v2_g10_dr_computer_q049_fa,fa,computer_science,Header & Footer در این درس بیشتر برای چه نوع ا...,9.0
10,bge_m3_dense,chunk_v2_g10_dr_computer_q004_ps,ps,computer_science,په ورډ کې نوې کرښې یا پاراګراف ته د تلو لپاره ...,10.0
97,bge_m3_dense,chunk_v2_g10_dr_computer_q033_ps,ps,computer_science,متن ولې انټرنېټ هغه عامل بولي چې نړۍ یې د یوې ...,6.0
124,bge_m3_dense,chunk_v2_g10_dr_computer_q042_ps,ps,computer_science,کوم انتخاب د حافظې رول په غیر مستقیمه خو د درس...,NaN
151,bge_m3_dense,chunk_v2_g10_dr_computer_q051_ps,ps,computer_science,کتاب د پاراګراف تنظیم تر ډېره د کوم هدف لپاره ...,NaN
157,bge_m3_dense,chunk_v2_g10_dr_computer_q053_ps,ps,computer_science,د دې درس په منطق کې ویب براوزر تر ډېره کوم رول...,6.0


,candidate_pool_size,query_id,language,expected_subject,query,first_relevant_rank_at_3,reranked_first_relevant_rank_at_top3
14,10,chunk_v2_g10_dr_computer_q005_en,en,computer_science,Which command is introduced for changing the p...,NaN,1.0
123,10,chunk_v2_g10_dr_computer_q042_fa,fa,computer_science,کدام گزینه نقش حافظه را به زبان غیرمستقیم اما ...,NaN,2.0
126,10,chunk_v2_g10_dr_computer_q043_fa,fa,computer_science,چرا در این درس CPU را «مغز کمپیوتر» می‌خوانند؟,NaN,2.0
141,10,chunk_v2_g10_dr_computer_q048_fa,fa,computer_science,کدام وضعیت، کاربرد فرمان Search را بهتر بازگو ...,NaN,1.0
165,10,chunk_v2_g10_dr_computer_q056_fa,fa,computer_science,کدام گزینه از نمونه‌های حافظه کمکی به شمار می‌...,NaN,2.0
127,10,chunk_v2_g10_dr_computer_q043_ps,ps,computer_science,ولې په دې درس کې CPU ته د «کمپیوټر مغز» ویل شوي؟,NaN,2.0
139,10,chunk_v2_g10_dr_computer_q047_ps,ps,computer_science,په ورډ کې د کار ساحه په یوه جمله کې څنګه راټول...,NaN,1.0
184,10,chunk_v2_g10_dr_computer_q062_ps,ps,computer_science,په Edit کې کوم امر د یوې کلمې د موندلو او له ب...,NaN,3.0
193,10,chunk_v2_g10_dr_computer_q065_ps,ps,computer_science,کوم انتخاب د لینکس د برتیاؤ په لړ کې نه دی یاد...,NaN,3.0


""


{'run_output_dir': '/home/nasher/Documents/projects/kankor-rag-space/data/experiments/benchmark_kankor_bge_m3/20260409T164725Z',
 'aggregate_comparison': '/home/nasher/Documents/projects/kankor-rag-space/data/experiments/benchmark_kankor_bge_m3/20260409T164725Z/aggregate_comparison.csv',
 'reranker_sweep_summary': '/home/nasher/Documents/projects/kankor-rag-space/data/experiments/benchmark_kankor_bge_m3/20260409T164725Z/reranker_sweep_summary.csv',
 'reranker_promotion_details': '/home/nasher/Documents/projects/kankor-rag-space/data/experiments/benchmark_kankor_bge_m3/20260409T164725Z/reranker_promotion_details.csv'}